<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training/4_10_model_tft_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4_10_model_tft

TFT: Temporal Fusion Transformer

## Introducción y Resumen

Temporal Fusion Transformer (TFT) fue propuesto por Google (Lim et al., 2020).
Está diseñado específicamente para series temporales multivariadas y combina lo mejor de varios mundos:

- LSTM → para capturar dependencias temporales locales.

- Self-Attention (Transformer) → para capturar relaciones de largo plazo y entre features.

- Gating + Variable Selection Networks → para seleccionar dinámicamente qué features son más relevantes en cada instante.

- Interpretabilidad → puedes visualizar la importancia temporal y por variable.

👉 En tu caso:

- Tienes ventanas fijas de 60 minutos (window_size=60).
- Cada ventana tiene muchas features (entre 900 y 1080), con relaciones complejas.
- Necesitas capturar patrones secuenciales y relevancia entre indicadores técnicos y alpha factors.

➡️ El TFT es ideal.


## 0. Configuración del Entorno


### 0.1. Instalación de librerías


In [1]:
# ==============================================
# 1) ELIMINAR TODO LO VIEJO
# ==============================================
!pip uninstall -y torch torchvision torchaudio xformers lightning pytorch-forecasting pytorch-lightning


# ==============================================
# 2) INSTALAR PYTORCH GPU (CUDA 12.1) + TORCHVISION + TORCHAUDIO
# Compatible con Colab + Python 3.12
# ==============================================
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1

# ==============================================
# 3) INSTALAR LIGHTNING MODERNO + PYTORCH FORECASTING MODERNO
# (Compatibles con Python 3.12 y con el nuevo Lightning)
# ==============================================
!pip install -q "lightning>=2.2.0" "pytorch-forecasting"

Found existing installation: torch 2.8.0+cu126
Uninstalling torch-2.8.0+cu126:
  Successfully uninstalled torch-2.8.0+cu126
Found existing installation: torchvision 0.23.0+cu126
Uninstalling torchvision-0.23.0+cu126:
  Successfully uninstalled torchvision-0.23.0+cu126
Found existing installation: torchaudio 2.8.0+cu126
Uninstalling torchaudio-2.8.0+cu126:
  Successfully uninstalled torchaudio-2.8.0+cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 298.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 274.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 251.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 412.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 445.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 329.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 245.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━

### 0.2. Importación de librerías


In [2]:
# ==============================
# Librerías de modelado (Lightning moderno)
# ==============================
import lightning.pytorch as pl
import torch

# ==============================
# PyTorch Forecasting (TFT y utilidades)
# ==============================
from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer,
)
from pytorch_forecasting.metrics import RMSE
from torch.utils.data import DataLoader

# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning tradicional
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

import joblib

warnings.filterwarnings("ignore")

In [3]:
import sys, platform
import numpy
import scipy
import sklearn
import torch
import lightning.pytorch as pl
import pytorch_forecasting

print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("Torch:", torch.__version__)
print("Lightning:", pl.__version__)
print("Forecasting:", pytorch_forecasting.__version__)

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.3
sklearn: 1.6.1
Torch: 2.5.1+cu121
Lightning: 2.5.6
Forecasting: 1.5.0


### 0.3. Acceso a Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [5]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [6]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [7]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [8]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [9]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_30 = features_dict["features_to_30"]
features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]


In [10]:
print(f'Listado de features para 30min ({len(features_to_30)}): {features_to_30}')
print(f'Listado de features para 60min ({len(features_to_60)}): {features_to_60}')
print(f'Listado de features para 90min ({len(features_to_90)}): {features_to_90}')

Listado de features para 30min (5): ['ire_90', 'rev_mom_z_90', 'roc_60', 'rev_score_90', 'price_ema30']
Listado de features para 60min (7): ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'rev_mom_vol_z_60', 'momentum_5', 'roc_20']
Listado de features para 90min (7): ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

**SE CARGAN LAS VENTANAS GENERADAS A PARTIR DE DATASET SUMSAMPLEADO AL 10%**

### 2.0. Funciones

#### Función para cargar ventanas

In [11]:
def load_windows_and_scaler(target: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz'
    path_valid  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz'
    path_test   = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/3_dataset_preparation/global_scaler_{target[0]+target[1]}.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### Función para revisar información de ventanas

In [12]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

    return  X_train.shape[0], X_valid.shape[0], X_test.shape[0]

In [13]:
features_base = ['open', 'high', 'low', 'close', 'volume']

features_30 = features_base + features_to_30
features_60 = features_base + features_to_60
features_90 = features_base + features_to_90

window_size = 90

### 2.1 Carga de ventanas 30 minutos

In [14]:
X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30, scaler_30 = load_windows_and_scaler(target = '30_ss_20')

**Con el 10% del dataset**


In [15]:
n_samples_train_30, n_samples_valid_30, n_samples_test_30 = xy_info( '30', X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30)

Información para horizonte de 30 minutos:

Set de entrenamiento:
	38613 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	38613 targets.
	Distribución y: mean=0.000070, std=0.002967, min=-0.028111, max=0.032372

Set de validación:
	8229 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	8229 targets.
	Distribución y: mean=0.000152, std=0.002932, min=-0.019471, max=0.017524

Set de testeo:
	8229 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	8229 targets.
	Distribución y: mean=0.000247, std=0.002222, min=-0.011682, max=0.009593


**Con el dataset completo 100%**

Información para horizonte de 30 minutos:

Set de entrenamiento:

      193487 ventanas (n_samples).
      900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
      193487 targets.
      Distribución y: mean=0.000056, std=0.002760, min=-0.028931, max=0.032372

Set de validación:

      41567 ventanas (n_samples).
      900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
      41567 targets.
      Distribución y: mean=0.000103, std=0.003273, min=-0.023167, max=0.068056

Set de testeo:

      41567 ventanas (n_samples).
      900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
      41567 targets.
      Distribución y: mean=-0.000023, std=0.002407, min=-0.015982, max=0.016097

### 2.2 Carga de ventanas 60 minutos

In [16]:
X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60, scaler_60 = load_windows_and_scaler(target = '60_ss_20')

In [17]:
n_samples_train_60, n_samples_valid_60, n_samples_test_60 = xy_info( '60', X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60)

Información para horizonte de 60 minutos:

Set de entrenamiento:
	38613 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	38613 targets.
	Distribución y: mean=0.000090, std=0.004149, min=-0.039316, max=0.036224

Set de validación:
	8229 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	8229 targets.
	Distribución y: mean=0.000300, std=0.003814, min=-0.020418, max=0.017340

Set de testeo:
	8229 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	8229 targets.
	Distribución y: mean=0.000459, std=0.003247, min=-0.015610, max=0.018561


### 2.3 Carga de ventanas 90 minutos

In [18]:
X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90_ss_20')

In [19]:
n_samples_train_90, n_samples_valid_90, n_samples_test_90 = xy_info('90', X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	38613 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	38613 targets.
	Distribución y: mean=0.000047, std=0.004910, min=-0.037742, max=0.028034

Set de validación:
	8229 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	8229 targets.
	Distribución y: mean=0.000459, std=0.004391, min=-0.024820, max=0.019050

Set de testeo:
	8229 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	8229 targets.
	Distribución y: mean=0.000596, std=0.004284, min=-0.015318, max=0.025985


## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [20]:
def load_metrics(data: str):
    data_path = f'{drive_path}/4_model_training/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [21]:
def metrics_verify(data: str) -> bool:
    data_path = f"{drive_path}/4_model_training/{data}.parquet"
    return os.path.exists(data_path)


In [22]:
def load_or_create_metrics (data:str):
  if metrics_verify(data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[4:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [23]:
tft_metrics, metrics = load_or_create_metrics("4_10_tft_metrics")

Las métricas no existen. Se crea el dataset _tft_metrics para almacenar las métricas


In [24]:
tft_metrics

,RMSE,MAE,R2,SMAPE,DirAcc


### 3.2. Función para guardar métricas

In [25]:
def save_metrics (metrics,  metrics_name: str):
  metrics_path = f"{drive_path}/4_model_training/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [26]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [27]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

In [28]:
tft_metrics

,RMSE,MAE,R2,SMAPE,DirAcc


## 4. Re-formateo más Encoder mínimo

Helper para re-formatear nuestras ventanas 2D a 3D que el formato que el modelo necesita.

### 4.1. Helper: de 2D (aplanado) a 3D (B, T, F)

Lo usamos para cada set y horizonte. Nuestro `window_size = 90` y los `n_features` depende del horizonte de tiempo. El siguiente código valida que `windows_size * n_features == X.shape[1]`

In [29]:
import numpy as np
import torch
import torch.nn as nn
import math

def reshape_windows(X_flat: np.ndarray, window_size: int, n_features: int) -> np.ndarray:
    """
    Convierte X de (N, window_size * n_features) a (N, window_size, n_features).
    Valida la consistencia del producto.
    """
    assert X_flat.ndim == 2, "Se esperaba X_flat con 2D (N, T*F)."
    N, TF = X_flat.shape
    assert window_size * n_features == TF, (
        f"Inconsistencia: {window_size} * {n_features} != {TF}"
    )
    return X_flat.reshape(N, window_size, n_features)

In [30]:
window_size = 90
features_base = ['open','high','close','low','volume']

In [31]:
n_features_30 = len (features_30)
n_features_60 = len (features_60)
n_features_90 = len (features_90)

In [32]:
# Re-shape de tus matrices 2D -> 3D
Xtr_30   =   reshape_windows(X_train_30_scaled, window_size, n_features_30)
Xva_30  =   reshape_windows(X_valid_30_scaled, window_size, n_features_30)
Xte_30  = reshape_windows(X_test_30_scaled, window_size, n_features_30)

Xtr_60   =   reshape_windows(X_train_60_scaled, window_size, n_features_60)
Xva_60  =   reshape_windows(X_valid_60_scaled, window_size, n_features_60)
Xte_60  = reshape_windows(X_test_60_scaled, window_size, n_features_60)

Xtr_90   =   reshape_windows(X_train_90_scaled, window_size, n_features_90)
Xva_90  =   reshape_windows(X_valid_90_scaled, window_size, n_features_90)
Xte_90  = reshape_windows(X_test_90_scaled, window_size, n_features_90)

In [33]:
ytr_30 = y_train_30
yva_30 = y_valid_30
yte_30 = y_test_30

ytr_60 = y_train_60
yva_60 = y_valid_60
yte_60 = y_test_60

ytr_90 = y_train_90
yva_90 = y_valid_90
yte_90 = y_test_90

In [34]:
# Comprobación de shapes
print('\nReshape de ventanas 30min:\n')
print(f'\tXtr_30.shape:\t{Xtr_30.shape}\t\tytr_30.shape:\t{ytr_30.shape}')
print(f'\tXva_30.shape:\t{Xva_30.shape}\t\tyva_30.shape:\t{yva_30.shape}')
print(f'\tXte_30.shape:\t{Xte_30.shape}\t\tyte_30.shape:\t{yte_30.shape}')


print('\nReshape de ventanas 60min:\n')
print(f'\tXtr_60.shape:\t{Xtr_60.shape}\t\tytr_60.shape:\t{ytr_60.shape}')
print(f'\tXva_60.shape:\t{Xva_60.shape}\t\tyva_60.shape:\t{yva_60.shape}')
print(f'\tXte_60.shape:\t{Xte_60.shape}\t\tyte_60.shape:\t{yte_60.shape}')


print('\nReshape de ventanas 90min:\n')
print(f'\tXtr_90.shape:\t{Xtr_90.shape}\t\tytr_90.shape:\t{ytr_90.shape}')
print(f'\tXva_90.shape:\t{Xva_90.shape}\t\tyva_90.shape:\t{yva_90.shape}')
print(f'\tXte_90.shape:\t{Xte_90.shape}\t\tyte_90.shape:\t{yte_90.shape}')



Reshape de ventanas 30min:

	Xtr_30.shape:	(38613, 90, 10)		ytr_30.shape:	(38613,)
	Xva_30.shape:	(8229, 90, 10)		yva_30.shape:	(8229,)
	Xte_30.shape:	(8229, 90, 10)		yte_30.shape:	(8229,)

Reshape de ventanas 60min:

	Xtr_60.shape:	(38613, 90, 12)		ytr_60.shape:	(38613,)
	Xva_60.shape:	(8229, 90, 12)		yva_60.shape:	(8229,)
	Xte_60.shape:	(8229, 90, 12)		yte_60.shape:	(8229,)

Reshape de ventanas 90min:

	Xtr_90.shape:	(38613, 90, 12)		ytr_90.shape:	(38613,)
	Xva_90.shape:	(8229, 90, 12)		yva_90.shape:	(8229,)
	Xte_90.shape:	(8229, 90, 12)		yte_90.shape:	(8229,)


## 5. Modelo TFT

El TFT “oficial” (pytorch-forecasting) exige un DataFrame con columnas especiales. Como ya tienes 3D, usa este helper para construirlo (por única vez):

In [35]:
import pandas as pd
import numpy as np

def build_tft_df(X, y, prefix="feat"):
    """
    X: (N, T, F)
    y: (N,)
    Devuelve un DataFrame long:
        - time_idx  : 0..T-1 (repetido por cada serie)
        - group_id  : id de ventana/serie (0..N-1)
        - target    : y repetida en todos los pasos de la serie (simple y directo)
        - feat_j    : columnas con las features
    """
    N, T, F = X.shape

    data = {
        "time_idx": np.tile(np.arange(T), N),
        "group_id": np.repeat(np.arange(N), T),
        "target":   np.repeat(y, T),
    }

    # aplanar features por columna
    for j in range(F):
        data[f"{prefix}_{j}"] = X[:, :, j].reshape(-1)

    df = pd.DataFrame(data)
    return df

In [36]:
# 30 min
df_tr_30 = build_tft_df(Xtr_30, ytr_30, prefix="f30")
df_va_30 = build_tft_df(Xva_30, yva_30, prefix="f30")

# 60 min
df_tr_60 = build_tft_df(Xtr_60, ytr_60, prefix="f60")
df_va_60 = build_tft_df(Xva_60, yva_60, prefix="f60")

# 90 min
df_tr_90 = build_tft_df(Xtr_90, ytr_90, prefix="f90")
df_va_90 = build_tft_df(Xva_90, yva_90, prefix="f90")

## 6. Crear listas para TimeSeriesDataSet:

Listas para los TimeSeriesDataset

In [37]:
#Para el horizonte de 30min:
feature_cols_30 = [c for c in df_tr_30.columns if c.startswith("f30_")]
time_varying_known_reals_30   = feature_cols_30
time_varying_unknown_reals_30 = ["target"]
static_reals_30         = []
static_categoricals_30  = []

#Para el horizonte de 60min:
feature_cols_60 = [c for c in df_tr_60.columns if c.startswith("f60_")]
time_varying_known_reals_60   = feature_cols_60
time_varying_unknown_reals_60 = ["target"]
static_reals_60         = []
static_categoricals_60  = []

#Para el horizonte de 90min:
feature_cols_90 = [c for c in df_tr_90.columns if c.startswith("f90_")]
time_varying_known_reals_90   = feature_cols_90
time_varying_unknown_reals_90 = ["target"]
static_reals_90         = []
static_categoricals_90  = []

## 7. Crear TimeSeriesDataSet:

In [39]:
max_encoder_length = 89      # tus 90 minutos de historial
max_prediction_length = 1    # retorno a 30 min como un único valor

### 1. Para 30 minutos:

In [40]:
training_30 = TimeSeriesDataSet(
    df_tr_30,
    time_idx="time_idx",
    target="target",
    group_ids=["group_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    time_varying_known_reals=time_varying_known_reals_30,
    time_varying_unknown_reals=time_varying_unknown_reals_30,
    static_reals=static_reals_30,
    static_categoricals=static_categoricals_30,

    target_normalizer=None,     # tus y ya están en escala razonable (retornos pequeños)
)

In [41]:
validation_30 = TimeSeriesDataSet.from_dataset(
    training_30,
    df_va_30,
    predict=False,
    stop_randomization=True,
)

### 2. Para 60 minutos:

In [42]:
training_60 = TimeSeriesDataSet(
    df_tr_60,
    time_idx="time_idx",
    target="target",
    group_ids=["group_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    time_varying_known_reals=time_varying_known_reals_60,
    time_varying_unknown_reals=time_varying_unknown_reals_60,
    static_reals=static_reals_60,
    static_categoricals=static_categoricals_60,

    target_normalizer=None,
)

In [43]:
validation_60 = TimeSeriesDataSet.from_dataset(
    training_60,
    df_va_60,
    predict=False,
    stop_randomization=True,
)

### 3. Para 90 minutos:

In [44]:
training_90 = TimeSeriesDataSet(
    df_tr_90,
    time_idx="time_idx",
    target="target",
    group_ids=["group_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    time_varying_known_reals=time_varying_known_reals_90,
    time_varying_unknown_reals=time_varying_unknown_reals_90,
    static_reals=static_reals_90,
    static_categoricals=static_categoricals_90,

    target_normalizer=None,
)

In [45]:
validation_90 = TimeSeriesDataSet.from_dataset(
    training_90,
    df_va_90,
    predict=False,
    stop_randomization=True,
)

## 8. Crear Dataloaders:

In [46]:
batch_size = 32


### 1. Para 30min

In [47]:
train_dataloader_30 = training_30.to_dataloader(
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

val_dataloader_30 = validation_30.to_dataloader(
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

### 2. Para 60min

In [48]:
train_dataloader_60 = training_60.to_dataloader(
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

val_dataloader_60 = validation_60.to_dataloader(
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

### 3. Para 90min

In [49]:
train_dataloader_90 = training_90.to_dataloader(
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

val_dataloader_90 = validation_90.to_dataloader(
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

## 9. Definir modelo TFT:

In [50]:
import torch

torch.backends.cudnn.enabled = False
print("cuDNN enabled:", torch.backends.cudnn.enabled)

cuDNN enabled: False


### 1. TFT para 30min

In [51]:
tft_30 = TemporalFusionTransformer.from_dataset(
    training_30,

    # Capacidad del modelo (moderada)
    hidden_size=16,                         # tamaño de las capas LSTM y de representación
    attention_head_size=2,            # pocas cabezas de atención para cuidar memoria
    hidden_continuous_size=8,     # tamaño de los MLP internos para features continuas
    lstm_layers=1,                            # empezamos con 1 capa LSTM (más estable en Colab)

    # Regularización
    dropout=0.1,                                 # moderado; se puede subir a 0.2 si ves overfitting

    # Optimización
    learning_rate=1e-3,
    loss=RMSE(),                                    # regresión sobre retornos
    reduce_on_plateau_patience=3,  # si la validación no mejora, baja el LR

    # Logging básico
    log_interval=10,
    log_val_interval=1,
)

### 2. TFT para 60min

In [52]:
tft_60 = TemporalFusionTransformer.from_dataset(
    training_60,
    hidden_size=16,
    attention_head_size=2,
    hidden_continuous_size=8,
    lstm_layers=1,

    dropout=0.1,
    learning_rate=1e-3,
    loss=RMSE(),
    reduce_on_plateau_patience=3,

    log_interval=10,
    log_val_interval=1,
)

### 3. TFT para 90min

In [53]:
tft_90 = TemporalFusionTransformer.from_dataset(
    training_90,
    hidden_size=16,
    attention_head_size=2,
    hidden_continuous_size=8,
    lstm_layers=1,

    dropout=0.1,
    learning_rate=1e-3,
    loss=RMSE(),
    reduce_on_plateau_patience=3,

    log_interval=10,
    log_val_interval=1,
)

## 10. Entrenamiento

### 1. Entrenamiento 30min

In [ ]:
#Primer Entrenamiento

trainer_30_0 = pl.Trainer(
    max_epochs=5,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    precision=32,             # Mantener 32 bits para estabilidad
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    log_every_n_steps=10,
)

trainer_30_0.fit(
    tft_30,
    train_dataloaders=train_dataloader_30,
    val_dataloaders=val_dataloader_30,
)


'\ntrainer_30 = pl.Trainer(\n    max_epochs=5,\n    accelerator="gpu" if torch.cuda.is_available() else "cpu",\n    precision=32,             # Mantener 32 bits para estabilidad\n    gradient_clip_val=0.1,\n    enable_checkpointing=False,\n    log_every_n_steps=10,\n)\n\ntrainer_30.fit(\n    tft_30,\n    train_dataloaders=train_dataloader_30,\n    val_dataloaders=val_dataloader_30,\n)\n'

In [54]:
#Early Stopping para segundo Entrenamiento

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
import torch

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min"
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

In [ ]:
#Segundo Entrenamiento
trainer_30_1 = pl.Trainer(
    max_epochs=30,              # en vez de 5
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    precision=32,
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    log_every_n_steps=10,
    callbacks=[early_stop, lr_monitor],
)

trainer_30_1.fit(
    tft_30,
    train_dataloaders=train_dataloader_30,
    val_dataloaders=val_dataloader_30,
)

print("Epoch actual:", trainer_30_1.current_epoch)
print("Épocas completadas:", trainer_30_1.fit_loop.epoch_progress.current.completed)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | RMSE                            | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 0      | train
3  | prescalers                         | ModuleDict                      | 176    | train
4  | static_variable_selection          | VariableSe

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

### 2. Entrenamiento 60min

In [ ]:
#Entrenamiento 1

trainer_60_0 = pl.Trainer(
    max_epochs=5,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    precision=32,             # Mantener 32 bits para estabilidad
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    log_every_n_steps=10,
)

trainer_60_0.fit(
    tft_60,
    train_dataloaders=train_dataloader_60,
    val_dataloaders=val_dataloader_60,
)


'\ntrainer_60 = pl.Trainer(\n    max_epochs=5,\n    accelerator="gpu" if torch.cuda.is_available() else "cpu",\n    precision=32,             # Mantener 32 bits para estabilidad\n    gradient_clip_val=0.1,\n    enable_checkpointing=False,\n    log_every_n_steps=10,\n)\n\ntrainer_60.fit(\n    tft_60,\n    train_dataloaders=train_dataloader_60,\n    val_dataloaders=val_dataloader_60,\n)\n'

In [ ]:
#Entrenamiento 2
trainer_60_1 = pl.Trainer(
    max_epochs=30,              # en vez de 5
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    precision=32,
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    log_every_n_steps=10,
    callbacks=[early_stop, lr_monitor],
)

trainer_60_1.fit(
    tft_60,
    train_dataloaders=train_dataloader_60,
    val_dataloaders=val_dataloader_60,
)

print("Epoch actual:", trainer_60_1.current_epoch)
print("Épocas completadas:", trainer_60_1.fit_loop.epoch_progress.current.completed)

### 3. Entrenamiento 90min

In [ ]:
#Entrenamiento 1

trainer_90_0 = pl.Trainer(
    max_epochs=5,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    precision=32,             # Mantener 32 bits para estabilidad
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    log_every_n_steps=10,
)

trainer_90_0.fit(
    tft_90,
    train_dataloaders=train_dataloader_90,
    val_dataloaders=val_dataloader_90,
)


'\ntrainer_90 = pl.Trainer(\n    max_epochs=5,\n    accelerator="gpu" if torch.cuda.is_available() else "cpu",\n    precision=32,             # Mantener 32 bits para estabilidad\n    gradient_clip_val=0.1,\n    enable_checkpointing=False,\n    log_every_n_steps=10,\n)\n\ntrainer_90.fit(\n    tft_90,\n    train_dataloaders=train_dataloader_90,\n    val_dataloaders=val_dataloader_90,\n)\n'

In [ ]:
#Entrenamiento 2
trainer_90_1 = pl.Trainer(
    max_epochs=30,              # en vez de 5
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    precision=32,
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    log_every_n_steps=10,
    callbacks=[early_stop, lr_monitor],
)


trainer_90_1.fit(
    tft_90,
    train_dataloaders=train_dataloader_90,
    val_dataloaders=val_dataloader_90,
)


print("Epoch actual:", trainer_90_1.current_epoch)
print("Épocas completadas:", trainer_90_1.fit_loop.epoch_progress.current.completed)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | RMSE                            | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 0      | train
3  | prescalers                         | ModuleDict                      | 208    | train
4  | static_variable_selection          | VariableSe

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch actual: 1
Épocas completadas: 1


## 11. Predicciones


### 1. Obtener predicciones

El horizonte h=30 es un solo valor agregado (retorno acumulado 30 min), lo normal es que el TFT tenga output_size=1 y se pueda hacer .ravel() sin problema.

In [ ]:
def eval_tft (tft_n, val_dataloader_n):

  tft_n.eval()
  y_true_list = []
  y_pred_list = []

  for batch_x, batch_y in iter(val_dataloader_n):
      with torch.no_grad():
          out = tft_n(batch_x)

      # Extraer el tensor de predicción del Output
      if hasattr(out, "prediction"):
          preds = out.prediction            # caso típico pytorch_forecasting
      elif isinstance(out, dict) and "prediction" in out:
          preds = out["prediction"]         # por si viene como dict
      else:
          preds = out                       # fallback: ya sería un Tensor

      # batch_y: (y, weight) o solo y
      if isinstance(batch_y, tuple):
          y = batch_y[0]
      else:
          y = batch_y

      y_true_list.append(y)
      y_pred_list.append(preds)

  # Concatenar y pasar a numpy
  y_true = torch.cat(y_true_list, dim=0).detach().cpu().numpy().ravel()
  y_pred = torch.cat(y_pred_list, dim=0).detach().cpu().numpy().ravel()

  return y_true, y_pred

### 2. Evaluar modelo

In [ ]:
y_true_30_0, y_pred_30_0 = eval_tft(tft_30_0, val_dataloader_30)

tft_30_metrics_0 = evaluate_model(
    model=None,
    X=None,
    y_true=y_true_30_0,
    y_pred=y_pred_30_0
)

print(tft_30_metrics_0)


{'RMSE': 0.002984879305586219, 'MAE': 0.0025317559484392405, 'R2': -0.3049823045730591, 'SMAPE': 131.0298309326172, 'DirAcc': 0.6254770992366412}


In [ ]:
y_true_30_1, y_pred_30_1 = eval_tft(tft_30_1, val_dataloader_30)

tft_30_metrics_1 = evaluate_model(
    model=None,
    X=None,
    y_true=y_true_30_1,
    y_pred=y_pred_30_1
)

print(tft_30_metrics_1)

In [ ]:
#Entrenamiento 1
tft_30_metrics_1 = {'RMSE': 0.002582161221653223, 'MAE': 0.00192218495067209, 'R2': 0.023397445678710938, 'SMAPE': 129.00155639648438, 'DirAcc': 0.6312022900763359}

#Entrenamiento 2
tft_30_metrics_2 = {'RMSE': 0.002984879305586219, 'MAE': 0.0025317559484392405, 'R2': -0.3049823045730591, 'SMAPE': 131.0298309326172, 'DirAcc': 0.6254770992366412}


In [ ]:
y_true_60, y_pred_60 = eval_tft(tft_60, val_dataloader_60)

tft_60_metrics = evaluate_model(
    model=None,
    X=None,
    y_true=y_true_60,
    y_pred=y_pred_60
)

print(tft_60_metrics)

{'RMSE': 0.005538965109735727, 'MAE': 0.004506876692175865, 'R2': -1.3762733936309814, 'SMAPE': 149.68658447265625, 'DirAcc': 0.4444179389312977}


In [ ]:
#Entrenamiento 1
tft_60_metrics_1 = {'RMSE': 0.003996796905994415, 'MAE': 0.0033854322973638773, 'R2': -0.23726654052734375, 'SMAPE': 132.96182250976562, 'DirAcc': 0.6548187022900763}

#Entrenamiento 2
tft_60_metrics_2 = {'RMSE': 0.005538965109735727, 'MAE': 0.004506876692175865, 'R2': -1.3762733936309814, 'SMAPE': 149.68658447265625, 'DirAcc': 0.4444179389312977}


In [ ]:
y_true_90, y_pred_90 = eval_tft(tft_90, val_dataloader_90)

tft_90_metrics = evaluate_model(
    model=None,
    X=None,
    y_true=y_true_90,
    y_pred=y_pred_90
)

print(tft_90_metrics)

{'RMSE': 0.004776321351528168, 'MAE': 0.003908372949808836, 'R2': -0.2322402000427246, 'SMAPE': 129.94976806640625, 'DirAcc': 0.6622137404580153}


In [ ]:
#Entrenamiento 1:
tft_90_metrics_1 = {'RMSE': 0.0021390877664089203, 'MAE': 0.002109619090333581, 'R2': 0.7528471946716309, 'SMAPE': 113.21489715576172, 'DirAcc': 0.7123091603053435}

#Entrenamiento 2:
tft_90_metrics_2 = {'RMSE': 0.004776321351528168, 'MAE': 0.003908372949808836, 'R2': -0.2322402000427246, 'SMAPE': 129.94976806640625, 'DirAcc': 0.6622137404580153}



In [ ]:
tft_metrics.loc['TFT_30_subsampleado_10%_1'] = tft_30_metrics_1
tft_metrics.loc['TFT_30_subsampleado_10%_2'] = tft_30_metrics_2

tft_metrics.loc['TFT_60_subsampleado_10%_1'] = tft_60_metrics_1
tft_metrics.loc['TFT_60_subsampleado_10%_2'] = tft_60_metrics_2

tft_metrics.loc['TFT_90_subsampleado_10%_1'] = tft_90_metrics_1
tft_metrics.loc['TFT_90_subsampleado_10%_2'] = tft_90_metrics_2


In [ ]:
tft_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
TFT_30_subsampleado_10%_1,0.002582,0.001922,0.023397,129.001556,0.631202
TFT_30_subsampleado_10%_2,0.002985,0.002532,-0.304982,131.029831,0.625477
TFT_60_subsampleado_10%_1,0.003997,0.003385,-0.237267,132.961823,0.654819
TFT_60_subsampleado_10%_2,0.005539,0.004507,-1.376273,149.686584,0.444418
TFT_90_subsampleado_10%_1,0.002139,0.002110,0.752847,113.214897,0.712309
TFT_90_subsampleado_10%_2,0.004776,0.003908,-0.232240,129.949768,0.662214


In [ ]:
save_metrics (tft_metrics, "4_10_tft_metrics")

Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_10_tft_metrics.parquet
